## Colab session configuration

Connect to google cloud runtime and use GPU.

## Reading hg38 genome assembly 

Import.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys, pathlib, subprocess
from pathlib import Path
!pip install pyfaidx
from pyfaidx import Fasta
from huggingface_hub import hf_hub_download

Configure root.

In [2]:
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/mini-gLM.git"
repo_dir = Path("mini-gLM")

if COLAB:
    root = Path("/content/mini-gLM")

    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])

else:
    root = Path.cwd().parent

sys.path.insert(0, str(root))

Try 512, then 1024 vocab size.

In [3]:
from src.tokenize import BPETokenizer

path = hf_hub_download(
    repo_id="eddykang06/hg38-pretraining",
    repo_type="dataset",
    filename="pretraining.csv.gz",
)

pretraining = pd.read_csv(path, compression = "gzip")

sequence_list = pretraining["sequence"].to_list()

tokenizer = BPETokenizer()
tokenized = tokenizer.train_tokenize(
    sequence_list = sequence_list,
    final_vocab_size = 512
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


pretraining.csv.gz:   0%|          | 0.00/772M [00:00<?, ?B/s]

KeyboardInterrupt: 

Write to parquet file, also get the merge rules and token ID mappings.

In [ ]:
import pickle

pretraining.drop(columns = ["sequence"], inplace= True)
pretraining["tokenized_sequence"] = tokenized

pretraining.to_parquet("tokenized-512.parquet", engine='pyarrow', compression='gzip', index = False)
merge_rules = tokenizer.merge_rules
token_to_idx = tokenizer.token_to_idx

with open("merge-rules-512.pkl", "wb") as f:
    pickle.dump(merge_rules, f)

with open("token-map-512.pkl", "wb") as f:
    pickle.dump(token_to_idx, f)

from google.colab import files

files.download("tokenized-512.parquet")
files.download("merge-rules-512.pkl")
files.download("token-map-512.pkl")

## Model architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Training

Implement masked language modeling training objective.
- Embedding module
- Relative positional embeddings
- Batching [B, L, D]
    - Token right padding to max length in batch
- Attention mask to ignore padding tokens
- Max sequence length cutoff
- Masked token prediction objective
Reference: https://huggingface.co/learn/llm-course/en/chapter6/5

Train tokenizer.

In [ ]:
from src.tokenize import BPETokenizer
from torch.utils.data import DataLoader, Dataset


class hg38Data(Dataset):
    """Custom dataset to load hg38 pre-training data from HuggingFace"""
    def __init__(self, tokenized_path):
        
        # Get sequences from HF path to tokenized dataset
        self.tokenized_path = tokenized_path

        # Convert the parquet file to a list of sequences lol
        self.tokenized_list 

    def __len__(self):
        len(self.tokenized_list)
    
    def __getitem__(self, idx):
        sequence = self.tokenized_list[idx]
        sequence = torch.tensor(sequence)

        return sequence


# Implement a data loader with dynamic batching and padding to max batch sequence lnegth
class DynamicDataLoader():
    """Custom DataLoader to implement Dynamic batching """

Implementing dynamic batching (batch similar sequences together)

- collate_fn
Dynamically pad each batch only to the longest sequence in that batch. PyTorch DataLoader directly supports custom collate_fn.
- pad_sequence
Pads a list of variable-length tensors into one batch tensor. The padded length becomes the longest sequence in that batch.
- custom batch_sampler
Groups examples by length or by total token count. DataLoader supports batch_sampler, which lets you control exactly which indices go into each batch.

In [ ]:
from torch.utils.data import DataLoader, Dataset


dataset = tokenized
train_loader = DataLoader    

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data import random_split


# Set seed
torch.manual_seed(111)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)